# 🔬 Programa: Machine Learning para la Toma de Decisiones
### Módulo 1 — Fundamentos Estadísticos: Pruebas de Hipótesis
**Objetivo:** Comprender, implementar e interpretar pruebas de hipótesis estadísticas y aplicarlas en contextos reales de Machine Learning.

---

## 🗺️ Contenido del Notebook

| # | Sección | Concepto Clave |
|---|---------|----------------|
| 1 | ¿Qué es una prueba de hipótesis? | Lógica del método científico |
| 2 | Errores Tipo I y Tipo II | α, β, Poder estadístico |
| 3 | El valor-p explicado | Malentendidos comunes |
| 4 | Pasos del proceso | Metodología paso a paso |
| 5 | Prueba Z | Medias con σ conocida |
| 6 | Prueba t (una muestra, dos muestras, pareada) | Medias con σ desconocida |
| 7 | Prueba Chi-Cuadrado | Variables categóricas |
| 8 | ANOVA | Comparar 3+ grupos |
| 9 | Pruebas no paramétricas | Sin supuestos de normalidad |
| 10 | Pruebas de hipótesis en ML | Feature selection, A/B testing, modelos |
| 11 | El problema de comparaciones múltiples | Corrección de Bonferroni / FDR |
| 12 | Tamaño del efecto | Cohen's d, η², Cramér's V |
| 13 | Bootstrap hypothesis testing | Inferencia sin distribuciones |

---

> 💡 **Para el aspirante a Data Scientist:** Las pruebas de hipótesis son el puente entre *intuición* y *evidencia*. Todo modelo de ML que despliegas en producción implica una apuesta estadística. Este notebook te dará las herramientas para apostar de forma informada.

In [ ]:
# ── Importaciones y configuración global ──────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import (
    norm, t as t_dist, chi2, f as f_dist,
    ttest_1samp, ttest_ind, ttest_rel,
    mannwhitneyu, kruskal, wilcoxon,
    shapiro, normaltest, levene,
    chi2_contingency, f_oneway,
    pearsonr, spearmanr
)
from statsmodels.stats.multitest import multipletests
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.rcParams.update({
    'figure.figsize':    (10, 5),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         12,
})
sns.set_palette("husl")
np.random.seed(42)

print("✅ Librerías cargadas correctamente")
print(f"   NumPy  {np.__version__}  |  Pandas {pd.__version__}")
print(f"   SciPy  {stats.__version__}")

---
## 1. ¿Qué es una Prueba de Hipótesis?

### 📖 La idea central

Una **prueba de hipótesis** es un procedimiento estadístico que usa datos muestrales para **tomar una decisión** sobre una afirmación acerca de una población.

**Analogía del juicio:**
> En un juicio, el acusado es **inocente hasta que se demuestre lo contrario**.  
> En estadística, la hipótesis nula (**H₀**) es "verdadera" hasta que los datos demuestren lo contrario.

---

### Componentes fundamentales

| Componente | Símbolo | Descripción |
|-----------|---------|-------------|
| Hipótesis nula | **H₀** | "No hay efecto / No hay diferencia" — la posición conservadora |
| Hipótesis alternativa | **H₁ o Hₐ** | "Sí hay efecto / Sí hay diferencia" — lo que queremos probar |
| Estadístico de prueba | **T** | Valor calculado de los datos que mide la evidencia |
| Valor-p | **p** | Probabilidad de obtener resultados tan extremos si H₀ fuera cierta |
| Nivel de significancia | **α** | Umbral de riesgo que aceptamos (típicamente 0.05) |

---

### Tipos de pruebas (por dirección)

```
H₀: μ = μ₀

H₁: μ ≠ μ₀  →  Prueba BILATERAL (dos colas)  — ¿diferente?
H₁: μ > μ₀  →  Prueba UNILATERAL derecha     — ¿mayor?
H₁: μ < μ₀  →  Prueba UNILATERAL izquierda   — ¿menor?
```

In [ ]:
# ── Visualizar la lógica de una prueba bilateral ──────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

x = np.linspace(-4, 4, 400)
y = norm.pdf(x)

ax.plot(x, y, 'steelblue', lw=2.5, label='Distribución bajo H₀')
ax.fill_between(x, y, where=(x <= -1.96), color='crimson', alpha=0.5, label='Zona de rechazo (α/2 = 2.5%)')
ax.fill_between(x, y, where=(x >= 1.96),  color='crimson', alpha=0.5)
ax.fill_between(x, y, where=((x > -1.96) & (x < 1.96)), color='steelblue', alpha=0.15, label='Zona de no rechazo (95%)')

# Estadístico de prueba de ejemplo
t_obs = 2.4
ax.axvline(t_obs, color='darkorange', lw=2.5, linestyle='--', label=f'Estadístico observado T = {t_obs}')
ax.annotate(f'T = {t_obs}\n(p < 0.05)', xy=(t_obs, 0.02),
            xytext=(t_obs + 0.3, 0.08), fontsize=11,
            arrowprops=dict(arrowstyle='->', color='darkorange'),
            color='darkorange', fontweight='bold')

ax.axvline(-1.96, color='gray', lw=1.5, linestyle=':', alpha=0.8)
ax.axvline( 1.96, color='gray', lw=1.5, linestyle=':', alpha=0.8)
ax.text(-1.96, 0.42, '-1.96', ha='center', color='gray', fontsize=10)
ax.text( 1.96, 0.42, '+1.96', ha='center', color='gray', fontsize=10)

ax.set_title('Prueba Bilateral: Lógica de Rechazo de H₀ (α = 0.05)', fontsize=14, fontweight='bold')
ax.set_xlabel('Estadístico de prueba (Z)', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

print("Interpretación:")
print(f"  Si |T| > 1.96  → Rechazamos H₀ (evidencia estadística suficiente)")
print(f"  Si |T| ≤ 1.96  → No rechazamos H₀ (evidencia insuficiente)")
print(f"\n  En este ejemplo: T = {t_obs} > 1.96  → ✅ Rechazamos H₀")

---
## 2. Errores Tipo I y Tipo II — El Dilema del Detective

### 📖 Concepto

Al tomar decisiones con datos imperfectos, **siempre existe la posibilidad de equivocarse** de dos formas distintas:

| | H₀ es **VERDADERA** | H₀ es **FALSA** |
|---|---|---|
| **Rechazamos H₀** | ❌ Error Tipo I (Falso Positivo) — α | ✅ Decisión Correcta — Potencia (1-β) |
| **No rechazamos H₀** | ✅ Decisión Correcta — (1-α) | ❌ Error Tipo II (Falso Negativo) — β |

---

### Analogía médica

| Situación | En diagnóstico médico | En ML / Data Science |
|-----------|----------------------|----------------------|
| Error Tipo I (α) | Diagnosticar cáncer a alguien sano | Detectar fraude donde no hay |
| Error Tipo II (β) | No detectar cáncer en un enfermo | Ignorar un fraude real |

### El trade-off fundamental

> **Reducir α → aumenta β** (y viceversa).  
> No podemos minimizar ambos errores al mismo tiempo con tamaño de muestra fijo.  
> La solución: **aumentar el tamaño de muestra** (más datos = más poder).

In [ ]:
# ── Visualizar Errores Tipo I, Tipo II y Poder estadístico ────────────────
fig, ax = plt.subplots(figsize=(12, 5))

x = np.linspace(-1, 8, 500)
mu0, mu1 = 0, 3          # Medias bajo H₀ y H₁
sigma = 1.5
alpha = 0.05
t_critico = norm.ppf(1 - alpha, loc=mu0, scale=sigma)  # ~2.47

y0 = norm.pdf(x, mu0, sigma)
y1 = norm.pdf(x, mu1, sigma)

# Distribuciones
ax.plot(x, y0, 'steelblue', lw=2.5, label='H₀ verdadera (μ₀ = 0)')
ax.plot(x, y1, 'tomato',    lw=2.5, label='H₁ verdadera (μ₁ = 3)')

# Error Tipo I: zona de rechazo cuando H₀ es cierta
ax.fill_between(x, y0, where=(x >= t_critico), color='crimson', alpha=0.4, label=f'Error Tipo I (α = {alpha})')

# Error Tipo II: zona de no rechazo cuando H₁ es cierta
ax.fill_between(x, y1, where=(x < t_critico),  color='gold',   alpha=0.5, label=f'Error Tipo II (β)')

# Poder estadístico
power = 1 - norm.cdf(t_critico, loc=mu1, scale=sigma)
ax.fill_between(x, y1, where=(x >= t_critico), color='limegreen', alpha=0.4, label=f'Poder = 1-β = {power:.2f}')

ax.axvline(t_critico, color='black', lw=1.5, linestyle='--', label=f'Valor crítico = {t_critico:.2f}')

ax.set_title('Errores Tipo I, Tipo II y Poder Estadístico', fontsize=14, fontweight='bold')
ax.set_xlabel('Valor del estadístico de prueba', fontsize=12)
ax.set_ylabel('Densidad de probabilidad', fontsize=12)
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

print("📊 Resumen del ejemplo:")
print(f"   Nivel de significancia (α)  : {alpha:.2f}  → Prob. Error Tipo I")
t_critico_val = norm.ppf(1 - alpha, loc=mu0, scale=sigma)
beta = norm.cdf(t_critico_val, loc=mu1, scale=sigma)
print(f"   Error Tipo II (β)           : {beta:.3f} → Prob. no detectar el efecto real")
print(f"   Poder estadístico (1-β)     : {power:.3f} → Prob. detectar el efecto si existe")
print(f"\n   ⚠️  Un poder < 0.80 se considera insuficiente en la mayoría de disciplinas")

---
## 3. El Valor-p — El Concepto más Malentendido en Estadística

### 📖 Definición exacta

> **El valor-p es la probabilidad de obtener un resultado tan extremo o más extremo que el observado, *asumiendo que H₀ es verdadera*.**

### ✅ Lo que SÍ significa p = 0.03
- Si H₀ fuera cierta, solo el 3% de las veces obtendríamos datos como los nuestros (o más extremos)
- Los datos son *inusuales* bajo H₀ → tenemos evidencia para rechazarla

### ❌ Lo que NO significa p = 0.03

| Mito | Realidad |
|------|----------|
| "H₀ tiene 3% de probabilidad de ser cierta" | El valor-p **no** es la probabilidad de que H₀ sea cierta |
| "H₁ tiene 97% de probabilidad de ser cierta" | ❌ Incorrecto — no mide eso |
| "p < 0.05 = resultado importante" | Un resultado puede ser significativo pero irrelevante (n muy grande) |
| "p > 0.05 = H₀ es verdadera" | Solo significa que no hay suficiente evidencia para rechazarla |

---

### Valores-p y tamaño de muestra: la trampa

$$p\text{-value} \downarrow \quad \text{cuando} \quad n \uparrow \quad \text{(todo se vuelve "significativo" con n grande)}$$

> 💡 **Lección para Data Scientists:** Con millones de filas (Big Data), casi cualquier diferencia será estadísticamente significativa. Por eso necesitas el **tamaño del efecto** (Sección 12).

In [ ]:
# ── Demostración: p-value vs tamaño de muestra ────────────────────────────
# Efecto real pero pequeño: μ₁ = 100.5, μ₀ = 100 (diferencia de solo 0.5)
mu_true = 100.5
mu_null = 100.0
sigma   = 5.0

sample_sizes = [10, 30, 50, 100, 300, 500, 1000, 5000, 10000]
p_values = []

for n in sample_sizes:
    sample = np.random.normal(mu_true, sigma, n)
    _, p = ttest_1samp(sample, popmean=mu_null)
    p_values.append(p)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: p-value vs n
ax1.plot(sample_sizes, p_values, 'o-', color='steelblue', lw=2, ms=7)
ax1.axhline(0.05, color='crimson', lw=2, linestyle='--', label='α = 0.05')
ax1.fill_between(sample_sizes, p_values, 0.05,
                 where=[p < 0.05 for p in p_values],
                 color='limegreen', alpha=0.2, label='Zona significativa')
ax1.set_xscale('log')
ax1.set_title('Efecto de n en el valor-p\n(diferencia real = 0.5 puntos)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tamaño de muestra (escala log)', fontsize=11)
ax1.set_ylabel('Valor-p', fontsize=11)
ax1.legend()

# Gráfico 2: distribuciones para n=50 vs n=5000
for n, color, label in [(50, 'steelblue', 'n=50'), (5000, 'tomato', 'n=5000')]:
    sample = np.random.normal(mu_true, sigma, n)
    _, p = ttest_1samp(sample, popmean=mu_null)
    ax2.hist(sample, bins=40, alpha=0.5, color=color, density=True,
             label=f'{label} | p={p:.4f}')

ax2.axvline(mu_null, color='black', lw=2, linestyle='--', label=f'H₀: μ = {mu_null}')
ax2.set_title('Distribución de muestras\n(mismo efecto, distinto n)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Valor', fontsize=11)
ax2.set_ylabel('Densidad', fontsize=11)
ax2.legend()

plt.tight_layout()
plt.show()

print("💡 Conclusión:")
print(f"   Con n=50  : el efecto de 0.5 puntos puede NO ser significativo")
print(f"   Con n=5000: el mismo efecto de 0.5 puntos SIEMPRE será significativo")
print(f"\n   → La significancia estadística ≠ importancia práctica")

---
## 4. Los 5 Pasos de una Prueba de Hipótesis

### 📋 Metodología estándar

Seguir estos pasos de forma sistemática evita errores de interpretación y asegura reproducibilidad:

```
PASO 1 ─ Formular las hipótesis
         H₀: [afirmación conservadora, con "="]
         H₁: [lo que quieres probar, con "≠", ">", "<"]

PASO 2 ─ Elegir el nivel de significancia (α)
         α = 0.05  (estándar en la mayoría de casos)
         α = 0.01  (cuando el error Tipo I es muy costoso)
         α = 0.10  (exploración, estudios piloto)

PASO 3 ─ Seleccionar y calcular el estadístico de prueba
         Depende del tipo de datos, tamaño de muestra y supuestos

PASO 4 ─ Calcular el valor-p (o valor crítico)
         p = P(T ≥ t_obs | H₀ verdadera)

PASO 5 ─ Tomar la decisión e interpretar
         Si p ≤ α → Rechazamos H₀ (resultado estadísticamente significativo)
         Si p > α → No rechazamos H₀ (evidencia insuficiente)
```

### 🔑 Elegir la prueba correcta

```
¿Cuántos grupos?
├── 1 grupo
│   ├── Datos continuos  → Prueba t de una muestra / Z
│   └── Datos binarios   → Prueba Z de proporciones
├── 2 grupos
│   ├── Independientes
│   │   ├── Paramétrico  → Prueba t de dos muestras independientes
│   │   └── No param.    → Mann-Whitney U
│   └── Pareados/Relacionados
│       ├── Paramétrico  → Prueba t pareada
│       └── No param.    → Wilcoxon signed-rank
└── 3+ grupos
    ├── Paramétrico      → ANOVA de una vía
    └── No paramétrico   → Kruskal-Wallis

¿Variables categóricas?
├── Independencia        → Chi-cuadrado
└── Bondad de ajuste     → Chi-cuadrado de bondad de ajuste
```

---
## 5. Prueba Z — Comparar una Media cuando σ es Conocida

### 📖 Cuándo usar la Prueba Z

- Tienes **una muestra** y quieres compararla con un valor de referencia conocido
- La **desviación estándar poblacional (σ) es conocida**, o n > 30 (por el TCL)
- Los datos son aproximadamente normales

$$Z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}}$$

### Caso práctico: Sistema de recomendación

> **Contexto:** Un e-commerce afirma que su nuevo algoritmo de recomendación genera en promedio **$85 de compra** por usuario. El estándar histórico es **$80** con σ = $15. Prueba con 50 usuarios.
> ¿Tiene el nuevo algoritmo un efecto significativo?

In [ ]:
# ── Prueba Z: Algoritmo de recomendación ──────────────────────────────────
# Parámetros del problema
mu_0  = 80     # Media bajo H₀ (histórico)
sigma = 15     # Desviación estándar poblacional (conocida)
n     = 50     # Tamaño de muestra
alpha = 0.05

# Simulamos los datos de los 50 usuarios con el nuevo algoritmo
np.random.seed(42)
compras = np.random.normal(loc=84, scale=sigma, size=n)  # media real ≈ 84

# ── PASO 1: Hipótesis ──────────────────────────────────────────────────────
print("=" * 55)
print("PASO 1: Formular hipótesis")
print("=" * 55)
print("  H₀: μ = 80  (el nuevo algoritmo no cambia el gasto)")
print("  H₁: μ ≠ 80  (el algoritmo sí tiene efecto — bilateral)")

# ── PASO 2: Nivel de significancia ────────────────────────────────────────
print(f"\nPASO 2: Nivel de significancia α = {alpha}")

# ── PASO 3: Estadístico Z ─────────────────────────────────────────────────
x_bar = np.mean(compras)
error_estandar = sigma / np.sqrt(n)
Z = (x_bar - mu_0) / error_estandar

print(f"\nPASO 3: Cálculo del estadístico Z")
print(f"  Media muestral  x̄ = {x_bar:.2f}")
print(f"  Error estándar  SE = σ/√n = {sigma}/√{n} = {error_estandar:.3f}")
print(f"  Z = (x̄ - μ₀) / SE = ({x_bar:.2f} - {mu_0}) / {error_estandar:.3f} = {Z:.3f}")

# ── PASO 4: Valor-p ───────────────────────────────────────────────────────
p_value = 2 * (1 - norm.cdf(abs(Z)))  # bilateral
z_critico = norm.ppf(1 - alpha / 2)

print(f"\nPASO 4: Valor-p")
print(f"  p-value (bilateral) = 2 × P(Z > |{Z:.3f}|) = {p_value:.4f}")
print(f"  Valor crítico Z     = ±{z_critico:.3f}")

# ── PASO 5: Decisión ──────────────────────────────────────────────────────
print(f"\nPASO 5: Decisión (α = {alpha})")
if p_value <= alpha:
    print(f"  p = {p_value:.4f} ≤ α = {alpha} → ✅ RECHAZAMOS H₀")
    print(f"  Conclusión: Existe evidencia estadística de que el nuevo")
    print(f"  algoritmo cambia el gasto promedio por usuario.")
else:
    print(f"  p = {p_value:.4f} > α = {alpha} → ❌ NO rechazamos H₀")

# ── Visualización ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x = np.linspace(-4, 4, 400)
ax.plot(x, norm.pdf(x), 'steelblue', lw=2.5)
ax.fill_between(x, norm.pdf(x), where=(x <= -z_critico), color='crimson', alpha=0.5, label=f'Zona rechazo α/2')
ax.fill_between(x, norm.pdf(x), where=(x >= z_critico),  color='crimson', alpha=0.5)
ax.axvline(Z,           color='darkorange', lw=2.5, linestyle='--', label=f'Z observado = {Z:.2f}')
ax.axvline(-z_critico,  color='gray', lw=1.5, linestyle=':', alpha=0.8)
ax.axvline( z_critico,  color='gray', lw=1.5, linestyle=':', alpha=0.8)
ax.set_title(f'Prueba Z — Algoritmo de Recomendación\np = {p_value:.4f} | {"Rechazamos H₀ ✅" if p_value <= alpha else "No rechazamos H₀"}',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Z', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Prueba t de Student — La Prueba más Usada en Data Science

### 📖 ¿Por qué existe la prueba t?

En la práctica, **casi nunca conocemos σ** (la desviación estándar de la población). La prueba t reemplaza σ con la desviación estándar de la muestra **s**, pagando un costo: la distribución t tiene **colas más pesadas** que la normal, lo que nos hace más conservadores.

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad \text{con } \nu = n-1 \text{ grados de libertad}$$

### Tres variantes de la prueba t

| Variante | Cuándo usar | Ejemplo en DS |
|----------|-------------|---------------|
| **Una muestra** | Comparar muestra con valor de referencia | ¿El modelo tiene accuracy > 85%? |
| **Dos muestras ind.** | Comparar dos grupos independientes | ¿Modelo A supera a Modelo B? |
| **Pareada** | Comparar antes/después en los mismos sujetos | ¿Mejoró el score después del fine-tuning? |

In [ ]:
# ── 6a. Prueba t de UNA MUESTRA ───────────────────────────────────────────
# Contexto: ¿El modelo de clasificación tiene accuracy superior al 75%?
# Realizamos 15 ejecuciones con distintos seeds → muestras de accuracy

np.random.seed(7)
accuracies = np.random.normal(loc=0.79, scale=0.04, size=15)
mu_0 = 0.75    # benchmark a superar
alpha = 0.05

# Test (H₁: μ > 0.75 → unilateral derecha → usamos alternative='greater')
t_stat, p_val = ttest_1samp(accuracies, popmean=mu_0, alternative='greater')

print("─" * 55)
print("6a. Prueba t — UNA MUESTRA")
print("─" * 55)
print(f"  H₀: μ_accuracy ≤ 0.75  (no supera el benchmark)")
print(f"  H₁: μ_accuracy > 0.75  (supera el benchmark)")
print(f"\n  Accuracies: {accuracies.round(4)}")
print(f"\n  n  = {len(accuracies)}")
print(f"  x̄  = {np.mean(accuracies):.4f}")
print(f"  s  = {np.std(accuracies, ddof=1):.4f}")
print(f"  t  = {t_stat:.3f}  |  p = {p_val:.4f}")
print(f"\n  {'✅ Rechazamos H₀: el modelo supera el 75%' if p_val < alpha else '❌ No rechazamos H₀: evidencia insuficiente'}")

print()
# ── 6b. Prueba t de DOS MUESTRAS INDEPENDIENTES ───────────────────────────
# Contexto: ¿El Modelo A tiene mejor F1-score que el Modelo B?
np.random.seed(42)
modelo_A = np.random.normal(loc=0.82, scale=0.03, size=20)
modelo_B = np.random.normal(loc=0.79, scale=0.035, size=20)

# Primero verificamos homocedasticidad (varianzas iguales)
levene_stat, levene_p = levene(modelo_A, modelo_B)
equal_var = levene_p > 0.05

t_stat2, p_val2 = ttest_ind(modelo_A, modelo_B, equal_var=equal_var, alternative='two-sided')

print("─" * 55)
print("6b. Prueba t — DOS MUESTRAS INDEPENDIENTES")
print("─" * 55)
print(f"  H₀: μ_A = μ_B  (modelos equivalentes)")
print(f"  H₁: μ_A ≠ μ_B  (modelos distintos — bilateral)")
print(f"\n  Levene (homocedasticidad): p = {levene_p:.4f} → {'varianzas iguales ✅' if equal_var else 'varianzas distintas — usamos Welch'}")
print(f"\n  Modelo A: x̄ = {np.mean(modelo_A):.4f}  |  Modelo B: x̄ = {np.mean(modelo_B):.4f}")
print(f"  t = {t_stat2:.3f}  |  p = {p_val2:.4f}")
print(f"\n  {'✅ Rechazamos H₀: los modelos son significativamente distintos' if p_val2 < alpha else '❌ No rechazamos H₀: diferencia no significativa'}")

print()
# ── 6c. Prueba t PAREADA ──────────────────────────────────────────────────
# Contexto: ¿Mejoró el F1 del modelo DESPUÉS del fine-tuning?
np.random.seed(0)
antes = np.random.normal(loc=0.74, scale=0.05, size=12)
despues = antes + np.random.normal(loc=0.04, scale=0.02, size=12)

t_stat3, p_val3 = ttest_rel(antes, despues, alternative='less')  # H₁: antes < despues

print("─" * 55)
print("6c. Prueba t — PAREADA (antes/después del fine-tuning)")
print("─" * 55)
print(f"  H₀: μ_diferencia = 0  (fine-tuning no mejora)")
print(f"  H₁: μ_antes < μ_despues  (fine-tuning mejora)")
print(f"\n  Diferencias (despues - antes): {(despues - antes).round(4)}")
print(f"  Media diferencia: {np.mean(despues - antes):.4f}")
print(f"  t = {t_stat3:.3f}  |  p = {p_val3:.4f}")
print(f"\n  {'✅ Rechazamos H₀: el fine-tuning mejoró el modelo significativamente' if p_val3 < alpha else '❌ No rechazamos H₀: mejora no significativa'}")

---
## 7. Prueba Chi-Cuadrado — Variables Categóricas

### 📖 Dos usos principales

**a) Prueba de independencia:** ¿Están relacionadas dos variables categóricas?  
$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

**b) Bondad de ajuste:** ¿Los datos siguen una distribución teórica esperada?

### Caso práctico: ¿El género influye en la adopción de un producto?

> Un equipo de producto analiza si el género del usuario es **independiente** de si adoptó o no una nueva feature.  
> Si son dependientes, deben crear campañas segmentadas.

In [ ]:
# ── Chi-cuadrado de independencia ─────────────────────────────────────────
# Tabla de contingencia: Género × Adoptó la feature
tabla = pd.DataFrame({
    'Adoptó': [120, 95],
    'No adoptó': [80, 105]
}, index=['Mujer', 'Hombre'])

print("Tabla de contingencia observada:")
print(tabla)
print(f"\nTotal usuarios: {tabla.values.sum()}")

# Ejecutar prueba Chi-cuadrado
chi2_stat, p_val, dof, expected = chi2_contingency(tabla)

print(f"\nFrecuencias esperadas bajo independencia:")
print(pd.DataFrame(expected.round(1), index=tabla.index, columns=tabla.columns))

print(f"\nResultados:")
print(f"  χ² = {chi2_stat:.4f}")
print(f"  Grados de libertad = {dof}")
print(f"  p-value = {p_val:.4f}")
print(f"\n  {'✅ Rechazamos H₀: género y adopción NO son independientes' if p_val < 0.05 else '❌ No rechazamos H₀: género y adopción son independientes'}")

# ── Visualización ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras agrupadas
tabla.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white', width=0.6)
axes[0].set_title('Frecuencias Observadas', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Cantidad de usuarios')
axes[0].set_xticklabels(tabla.index, rotation=0)
axes[0].legend(title='Feature')

# Mapa de calor de residuos estandarizados
residuos = (tabla.values - expected) / np.sqrt(expected)
im = axes[1].imshow(residuos, cmap='RdYlGn', vmin=-3, vmax=3)
axes[1].set_xticks(range(2)); axes[1].set_yticks(range(2))
axes[1].set_xticklabels(tabla.columns)
axes[1].set_yticklabels(tabla.index)
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f'{residuos[i, j]:.2f}', ha='center', va='center',
                     fontsize=14, fontweight='bold',
                     color='black' if abs(residuos[i, j]) < 2 else 'white')
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Residuos Estandarizados\n(|r| > 2 indica asociación fuerte)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Cramér's V (tamaño del efecto)
n = tabla.values.sum()
cramers_v = np.sqrt(chi2_stat / (n * (min(tabla.shape) - 1)))
print(f"\n📏 Tamaño del efecto — Cramér's V = {cramers_v:.3f}")
print(f"   (0.1=pequeño, 0.3=mediano, 0.5+=grande)")

---
## 8. ANOVA — Comparar 3 o más Grupos

### 📖 ¿Por qué no hacer múltiples pruebas t?

Si tienes 4 grupos (A, B, C, D) y haces todas las combinaciones de pruebas t (A-B, A-C, A-D, B-C, B-D, C-D = **6 pruebas**), la probabilidad de cometer al menos un Error Tipo I crece:

$$P(\text{al menos un falso positivo}) = 1 - (1 - \alpha)^k = 1 - (0.95)^6 \approx 0.26$$

**¡Un 26% de chances de encontrar diferencias falsas!** ANOVA mantiene α = 0.05 en una sola prueba.

### La lógica de ANOVA

$$F = \frac{\text{Varianza ENTRE grupos}}{\text{Varianza DENTRO de grupos}} = \frac{MS_{between}}{MS_{within}}$$

> Si F >> 1: los grupos son más distintos entre sí de lo que varían internamente → evidencia contra H₀

### Caso práctico: Comparar 4 estrategias de feature engineering

> ¿Las 4 estrategias producen diferente performance (RMSE) en validación?

In [ ]:
# ── ANOVA: Comparar 4 estrategias de feature engineering ─────────────────
np.random.seed(1)
estrategia_A = np.random.normal(loc=12.5, scale=1.2, size=25)  # baseline
estrategia_B = np.random.normal(loc=11.8, scale=1.1, size=25)  # normalización
estrategia_C = np.random.normal(loc=10.9, scale=1.3, size=25)  # PCA
estrategia_D = np.random.normal(loc=10.5, scale=1.0, size=25)  # embeddings

grupos = [estrategia_A, estrategia_B, estrategia_C, estrategia_D]
nombres = ['A: Baseline', 'B: Normaliz.', 'C: PCA', 'D: Embeddings']

# ── ANOVA de una vía ───────────────────────────────────────────────────────
F_stat, p_val = f_oneway(*grupos)

print("=" * 55)
print("ANOVA de una vía — Comparación de estrategias (RMSE)")
print("=" * 55)
print("  H₀: μ_A = μ_B = μ_C = μ_D  (estrategias equivalentes)")
print("  H₁: Al menos una estrategia difiere")
for nombre, grupo in zip(nombres, grupos):
    print(f"\n  {nombre}: x̄ = {np.mean(grupo):.3f}  s = {np.std(grupo, ddof=1):.3f}")
print(f"\n  F-estadístico = {F_stat:.3f}")
print(f"  p-value       = {p_val:.6f}")
print(f"\n  {'✅ Rechazamos H₀: hay diferencias significativas entre estrategias' if p_val < 0.05 else '❌ No rechazamos H₀'}")

# ── Post-hoc: Pruebas t con corrección de Bonferroni ─────────────────────
if p_val < 0.05:
    from itertools import combinations
    print("\n─ Post-hoc (Bonferroni) — ¿cuáles pares difieren? ─")
    pares = list(combinations(range(4), 2))
    p_values_posthoc = []
    for i, j in pares:
        _, p = ttest_ind(grupos[i], grupos[j])
        p_values_posthoc.append(p)

    # Corrección de Bonferroni
    reject, p_adj, _, _ = multipletests(p_values_posthoc, method='bonferroni')
    for (i, j), p_raw, p_corr, sig in zip(pares, p_values_posthoc, p_adj, reject):
        status = "✅ SIG" if sig else "  n.s."
        print(f"  {nombres[i]} vs {nombres[j]}: p_raw={p_raw:.4f}  p_adj={p_corr:.4f}  {status}")

# ── Boxplot comparativo ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(grupos, labels=nombres, patch_artist=True, notch=True,
                medianprops=dict(color='black', lw=2))
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title(f'ANOVA: RMSE por estrategia de Feature Engineering\nF = {F_stat:.2f}, p = {p_val:.6f}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('RMSE (error de validación)', fontsize=12)
ax.set_xlabel('Estrategia', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9. Pruebas No Paramétricas — Cuando los Datos No Son Normales

### 📖 ¿Cuándo usar pruebas no paramétricas?

- Los datos **no son normales** (test de Shapiro-Wilk falla)
- La muestra es **pequeña** (n < 30) y no puedes asumir normalidad
- Los datos son **ordinales** (escalas de 1-5, rankings)
- Hay **outliers extremos** que distorsionan las medias

### Equivalencias con pruebas paramétricas

| Paramétrica | No Paramétrica | Cuándo |
|-------------|----------------|--------|
| t de una muestra | Wilcoxon signed-rank | Datos no normales, n pequeño |
| t de dos muestras independientes | Mann-Whitney U | Distribuciones asimétricas |
| ANOVA de una vía | Kruskal-Wallis | No normalidad en 3+ grupos |
| Correlación de Pearson | Correlación de Spearman | Datos ordinales o con outliers |

> ⚠️ **No confundir:** Las pruebas no paramétricas **sí tienen supuestos** (independencia, escala ordinal/continua). No son "libres de supuestos".

In [ ]:
# ── 9a. Verificar normalidad antes de elegir la prueba ───────────────────
np.random.seed(99)
# Datos de ratings de usuarios (distribución asimétrica, no normal)
ratings_grupo_A = np.random.exponential(scale=3, size=30).clip(1, 10)
ratings_grupo_B = np.random.exponential(scale=2, size=30).clip(1, 10)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, datos, nombre, color in zip(axes[:2],
                                    [ratings_grupo_A, ratings_grupo_B],
                                    ['Grupo A', 'Grupo B'],
                                    ['steelblue', 'tomato']):
    ax.hist(datos, bins=12, color=color, alpha=0.7, edgecolor='white')
    stat_sw, p_sw = shapiro(datos)
    ax.set_title(f'{nombre}\nShapiro-Wilk: p = {p_sw:.4f}\n{"⚠️ No normal" if p_sw < 0.05 else "✅ Normal"}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Rating')
    ax.set_ylabel('Frecuencia')

# QQ-plot del Grupo A
from scipy.stats import probplot
probplot(ratings_grupo_A, plot=axes[2])
axes[2].set_title('Q-Q Plot — Grupo A\n(puntos fuera de la línea → no normal)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("─" * 55)
print("Test de normalidad (Shapiro-Wilk)")
print("─" * 55)
for datos, nombre in [(ratings_grupo_A, 'Grupo A'), (ratings_grupo_B, 'Grupo B')]:
    stat, p = shapiro(datos)
    print(f"  {nombre}: W = {stat:.4f}, p = {p:.4f} → {'⚠️ No normal → usar Mann-Whitney' if p < 0.05 else '✅ Normal → usar prueba t'}")

# ── 9b. Mann-Whitney U: comparación no paramétrica de dos grupos ──────────
u_stat, p_mw = mannwhitneyu(ratings_grupo_A, ratings_grupo_B, alternative='two-sided')

print(f"\n─ Mann-Whitney U ─")
print(f"  H₀: distribuciones iguales  |  H₁: distribuciones distintas")
print(f"  U = {u_stat:.1f}  |  p = {p_mw:.4f}")
print(f"  {'✅ Rechazamos H₀: los grupos tienen distribuciones distintas' if p_mw < 0.05 else '❌ No rechazamos H₀'}")

# Comparación con t (que sería incorrecto aquí)
_, p_t = ttest_ind(ratings_grupo_A, ratings_grupo_B)
print(f"\n  📌 Comparación instructiva:")
print(f"     Mann-Whitney p = {p_mw:.4f}  ← correcto para datos no normales")
print(f"     Prueba t      p = {p_t:.4f}  ← incorrecto (viola supuesto de normalidad)")

---
## 10. Pruebas de Hipótesis en Machine Learning

### 📖 ¿Dónde aparece la estadística en ML?

| Contexto en ML | Prueba apropiada | Pregunta |
|----------------|------------------|----------|
| Comparar dos modelos | Prueba t pareada / McNemar | ¿Modelo A supera a B significativamente? |
| Selección de features | Prueba t / ANOVA / Chi-cuadrado | ¿Esta feature está relacionada con el target? |
| A/B Testing (producción) | Prueba Z / t de proporciones | ¿El experimento mejoró la conversión? |
| Validar supuestos del modelo | Shapiro-Wilk / Levene / DW | ¿Los residuos son normales / homocedásticos? |
| Detección de data drift | KS test / Chi-cuadrado | ¿Los datos de producción cambiaron? |

---

### 10a. Feature Selection con pruebas estadísticas

> Usando el dataset Iris: ¿cuáles features discriminan mejor las especies?  
> Usaremos **ANOVA** para numéricas y **Chi-cuadrado** para categóricas.

In [ ]:
# ── 10a. Feature Selection con ANOVA — Dataset Iris ──────────────────────
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target
df_iris['especie'] = df_iris['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

feature_names = iris.feature_names
resultados = []

for feature in feature_names:
    grupos_por_clase = [
        df_iris[df_iris['target'] == cls][feature].values
        for cls in [0, 1, 2]
    ]
    F, p = f_oneway(*grupos_por_clase)
    # Eta-cuadrado: proporción de varianza explicada por la clase
    grand_mean = df_iris[feature].mean()
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in grupos_por_clase)
    ss_total   = sum((x - grand_mean)**2 for g in grupos_por_clase for x in g)
    eta2 = ss_between / ss_total
    resultados.append({'Feature': feature, 'F-stat': F, 'p-value': p, 'Eta²': eta2})

df_features = pd.DataFrame(resultados).sort_values('F-stat', ascending=False)
df_features['Significativa'] = df_features['p-value'].apply(lambda p: '✅' if p < 0.05 else '❌')
print("Feature Selection por ANOVA (α = 0.05):")
print(df_features.round(4).to_string(index=False))

# ── Visualización: F-stat de cada feature ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['limegreen' if p < 0.05 else 'tomato' for p in df_features['p-value']]
axes[0].barh(df_features['Feature'], df_features['F-stat'], color=colors)
axes[0].set_title('F-estadístico por Feature\n(verde = significativa)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('F-estadístico (más alto = más discriminativo)')

# Violin plot de la mejor feature
best_feature = df_features.iloc[0]['Feature']
sns.violinplot(data=df_iris, x='especie', y=best_feature, ax=axes[1],
               palette=['#4C72B0', '#55A868', '#C44E52'], inner='quartile')
axes[1].set_title(f'Distribución de "{best_feature}"\n(la feature más discriminativa)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Especie')

plt.tight_layout()
plt.show()

### 10b. Comparación de Modelos con Prueba t Pareada

> **Pregunta crítica de producción:** ¿El modelo B (Random Forest) es significativamente mejor que el modelo A (Logistic Regression) en el dataset de cáncer de mama?

Usamos **validación cruzada** con múltiples folds y comparamos las métricas fold-a-fold (prueba t pareada), ya que los mismos datos generan ambas métricas.

In [ ]:
# ── 10b. Comparar modelos con validación cruzada + prueba t pareada ───────
cancer = load_breast_cancer()
X, y   = cancer.data, cancer.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo_lr = LogisticRegression(max_iter=1000, random_state=42)
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

cv_folds = 10
scores_lr = cross_val_score(modelo_lr, X_scaled, y, cv=cv_folds, scoring='f1')
scores_rf = cross_val_score(modelo_rf, X_scaled, y, cv=cv_folds, scoring='f1')

print("─" * 60)
print("Comparación de modelos — F1 score por fold (CV=10)")
print("─" * 60)
df_comp = pd.DataFrame({'Fold': range(1, cv_folds+1),
                        'Logistic Regression': scores_lr.round(4),
                        'Random Forest': scores_rf.round(4),
                        'Diferencia (RF-LR)': (scores_rf - scores_lr).round(4)})
print(df_comp.to_string(index=False))
print(f"\n  LR: x̄ = {scores_lr.mean():.4f}  ±  {scores_lr.std():.4f}")
print(f"  RF: x̄ = {scores_rf.mean():.4f}  ±  {scores_rf.std():.4f}")

# Prueba t pareada (H₁: RF > LR)
t_stat, p_val = ttest_rel(scores_rf, scores_lr, alternative='greater')
print(f"\n  Prueba t pareada (H₁: RF > LR):")
print(f"  t = {t_stat:.3f}  |  p = {p_val:.4f}")
if p_val < 0.05:
    print(f"  ✅ Random Forest es significativamente mejor que Logistic Regression (α=0.05)")
else:
    print(f"  ❌ No hay evidencia estadística de que RF supere a LR")

# ── Visualización ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

folds_x = range(1, cv_folds + 1)
ax1.plot(folds_x, scores_lr, 'o-', color='steelblue', lw=2, ms=7, label='Logistic Regression')
ax1.plot(folds_x, scores_rf, 's-', color='tomato',    lw=2, ms=7, label='Random Forest')
ax1.fill_between(folds_x, scores_lr, scores_rf,
                 where=(scores_rf >= scores_lr), color='limegreen', alpha=0.2, label='RF mejor')
ax1.fill_between(folds_x, scores_lr, scores_rf,
                 where=(scores_rf < scores_lr),  color='salmon',    alpha=0.2, label='LR mejor')
ax1.set_title(f'F1 por Fold\nt={t_stat:.2f}, p={p_val:.4f}', fontsize=12, fontweight='bold')
ax1.set_xlabel('Fold'); ax1.set_ylabel('F1 Score')
ax1.legend(); ax1.set_xticks(list(folds_x))

ax2.boxplot([scores_lr, scores_rf], labels=['Logistic\nRegression', 'Random\nForest'],
            patch_artist=True, notch=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6),
            medianprops=dict(color='black', lw=2))
bp2 = ax2.boxplot([scores_lr, scores_rf], labels=['Logistic\nRegression', 'Random\nForest'],
                  patch_artist=True, notch=True,
                  medianprops=dict(color='black', lw=2))
bp2['boxes'][0].set_facecolor('steelblue'); bp2['boxes'][0].set_alpha(0.6)
bp2['boxes'][1].set_facecolor('tomato');    bp2['boxes'][1].set_alpha(0.6)
ax2.set_title('Distribución de F1\n(notch indica IC del 95% de la mediana)',
              fontsize=12, fontweight='bold')
ax2.set_ylabel('F1 Score')

plt.tight_layout()
plt.show()

### 10c. A/B Testing — La Prueba de Hipótesis en Producción

> **Escenario real:** Tu equipo lanzó una nueva versión del botón de compra (versión B). El equipo de producto quiere saber si la tasa de conversión mejoró con respecto a la versión original (versión A).

Este es el **experimento más común en empresas de tecnología**. Netflix, Uber, Airbnb y Spotify ejecutan cientos de A/B tests simultáneamente.

In [ ]:
# ── A/B Testing: Tasa de conversión (prueba Z de proporciones) ────────────
# Datos del experimento
n_A, conversiones_A = 4500, 432   # Versión A (control): 9.6%
n_B, conversiones_B = 4500, 518   # Versión B (tratamiento): 11.5%

p_A = conversiones_A / n_A
p_B = conversiones_B / n_B

# Estimación de la proporción combinada bajo H₀
p_pool = (conversiones_A + conversiones_B) / (n_A + n_B)

# Estadístico Z para diferencia de proporciones
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
Z  = (p_B - p_A) / se

# p-value unilateral (H₁: p_B > p_A)
p_val = 1 - norm.cdf(Z)

# Intervalo de confianza del 95% para la diferencia
diff     = p_B - p_A
se_diff  = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
ci_lower = diff - 1.96 * se_diff
ci_upper = diff + 1.96 * se_diff

print("=" * 60)
print("A/B TEST — Tasa de conversión del botón de compra")
print("=" * 60)
print(f"  Versión A (control)   : {conversiones_A}/{n_A} = {p_A:.2%}")
print(f"  Versión B (tratamiento): {conversiones_B}/{n_B} = {p_B:.2%}")
print(f"\n  H₀: p_B ≤ p_A  (B no mejora la conversión)")
print(f"  H₁: p_B > p_A  (B mejora la conversión)")
print(f"\n  Diferencia observada  : {diff:.4f} ({diff:.2%})")
print(f"  Incremento relativo   : {(p_B-p_A)/p_A:.2%} de mejora")
print(f"  IC 95% de diferencia  : [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"\n  Z = {Z:.3f}  |  p-value = {p_val:.4f}")
if p_val < 0.05:
    print(f"\n  ✅ RESULTADO: La versión B mejora significativamente la conversión")
    print(f"     Podemos lanzar la versión B a producción con confianza estadística")
    print(f"\n  💡 Impacto estimado: ~{int((p_B - p_A) * 10000)} conversiones adicionales")
    print(f"     por cada 10,000 usuarios")
else:
    print(f"\n  ❌ RESULTADO: No hay evidencia suficiente para preferir la versión B")

# ── Visualización ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Barras de conversión
versiones = ['Versión A\n(Control)', 'Versión B\n(Tratamiento)']
tasas     = [p_A, p_B]
colores   = ['steelblue', 'limegreen' if p_val < 0.05 else 'tomato']
bars = ax1.bar(versiones, [t * 100 for t in tasas], color=colores, width=0.45, edgecolor='white')
for bar, tasa in zip(bars, tasas):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{tasa:.2%}', ha='center', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(tasas) * 100 * 1.25)
ax1.set_ylabel('Tasa de conversión (%)', fontsize=11)
ax1.set_title(f'A/B Test — Tasa de Conversión\np = {p_val:.4f}', fontsize=12, fontweight='bold')

# Distribución bajo H₀ vs diferencia observada
x = np.linspace(-0.02, 0.04, 400)
se_h0 = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
ax2.plot(x, norm.pdf(x, 0, se_diff), 'steelblue', lw=2.5, label='Distribución bajo H₀')
ax2.fill_between(x, norm.pdf(x, 0, se_diff), where=(x >= diff), color='limegreen', alpha=0.4, label=f'p-value = {p_val:.4f}')
ax2.axvline(diff, color='darkorange', lw=2.5, linestyle='--', label=f'Diferencia obs. = {diff:.4f}')
ax2.axvline(0, color='gray', lw=1, linestyle=':')
ax2.set_title('Diferencia de proporciones bajo H₀', fontsize=12, fontweight='bold')
ax2.set_xlabel('p_B - p_A')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 11. El Problema de Comparaciones Múltiples

### 📖 El problema del "cherry picking" estadístico

Si haces **m pruebas de hipótesis independientes** cada una con α = 0.05, la probabilidad de obtener al menos un falso positivo es:

$$P(\text{≥ 1 error Tipo I}) = 1 - (1 - \alpha)^m$$

Con m = 20 pruebas: $1 - (0.95)^{20} \approx 0.64$ → ¡64% de chances de un falso positivo!

### Dos correcciones principales

| Método | Fórmula | Cuándo usar |
|--------|---------|-------------|
| **Bonferroni** | $\alpha_{ajustado} = \alpha / m$ | Conservador, pocas pruebas |
| **Benjamini-Hochberg (FDR)** | Controla la tasa de falsos descubrimientos | Muchas pruebas (genómica, Big Data) |

### Relevancia en ML

- Búsqueda de hiperparámetros: probar 50 combinaciones → inflación del α
- Feature selection automática: probar 100 features → muchos falsos positivos
- Análisis de subgrupos: comparar el modelo en muchos segmentos

In [ ]:
# ── Comparaciones múltiples: sin corrección vs Bonferroni vs FDR ──────────
np.random.seed(123)
m = 50   # Número de features a evaluar

# Simulamos: solo 5 features tienen efecto real, las otras 45 son ruido
target = np.random.normal(0, 1, 200)
features_reales  = [np.random.normal(0.5, 1, 200) for _ in range(5)]   # correlación real
features_ruido   = [np.random.normal(0,   1, 200) for _ in range(45)]  # sin correlación
todas_features   = features_reales + features_ruido
es_real          = [True]*5 + [False]*45

p_values_raw = []
for feature in todas_features:
    _, p = ttest_ind(feature[target > 0], feature[target <= 0])
    p_values_raw.append(p)

p_values_raw = np.array(p_values_raw)

# Correcciones
_, p_bonf, _, _ = multipletests(p_values_raw, method='bonferroni')
_, p_fdr,  _, _ = multipletests(p_values_raw, method='fdr_bh')

alpha = 0.05
sig_sin_corr = p_values_raw < alpha
sig_bonf     = p_bonf < alpha
sig_fdr      = p_fdr  < alpha

# Contar falsos positivos
fp_sin = sum(sig_sin_corr[5:])   # significativas en las 45 de ruido
fp_bon = sum(sig_bonf[5:])
fp_fdr = sum(sig_fdr[5:])

vp_sin = sum(sig_sin_corr[:5])   # verdaderos positivos (las 5 reales)
vp_bon = sum(sig_bonf[:5])
vp_fdr = sum(sig_fdr[:5])

print("=" * 60)
print(f"Comparaciones múltiples: {m} features, α = {alpha}")
print(f"Features con efecto real: 5  |  Features de ruido: 45")
print("=" * 60)
print(f"\n{'Método':<22}{'VP (de 5)':<14}{'FP (de 45)':<14}{'Total signif.'}")
print("-" * 60)
print(f"{'Sin corrección':<22}{vp_sin:<14}{fp_sin:<14}{sum(sig_sin_corr)}")
print(f"{'Bonferroni':<22}{vp_bon:<14}{fp_bon:<14}{sum(sig_bonf)}")
print(f"{'FDR (Benjamini-H.)':<22}{vp_fdr:<14}{fp_fdr:<14}{sum(sig_fdr)}")

# ── Visualización ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
titles = ['Sin corrección', 'Bonferroni', 'FDR (Benjamini-H.)']
p_sets = [p_values_raw, p_bonf, p_fdr]

for ax, title, p_set in zip(axes, titles, p_sets):
    colors = []
    for i, (p, real) in enumerate(zip(p_set, es_real)):
        if p < alpha and real:     colors.append('#2ecc71')  # Verdadero positivo
        elif p < alpha and not real: colors.append('#e74c3c')  # Falso positivo
        elif p >= alpha and real:  colors.append('#f39c12')  # Falso negativo
        else:                      colors.append('#95a5a6')  # Verdadero negativo

    ax.scatter(range(m), -np.log10(p_set), c=colors, s=60, zorder=5)
    ax.axhline(-np.log10(alpha), color='black', lw=1.5, linestyle='--', label=f'α = {alpha}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Índice de feature')
    ax.set_ylabel('-log₁₀(p-value)')

legend_elements = [
    mpatches.Patch(color='#2ecc71', label='Verdadero positivo'),
    mpatches.Patch(color='#e74c3c', label='Falso positivo'),
    mpatches.Patch(color='#f39c12', label='Falso negativo'),
    mpatches.Patch(color='#95a5a6', label='Verdadero negativo'),
]
axes[2].legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Conclusión:")
print(f"   Sin corrección: {fp_sin} falsos positivos → enviamos ruido al modelo")
print(f"   Bonferroni    : {fp_bon} falsos positivos → conservador, puede perder señales reales")
print(f"   FDR           : {fp_fdr} falsos positivos → balance entre sensibilidad y precisión")

---
## 12. Tamaño del Efecto — La Relevancia Práctica

### 📖 ¿Por qué no basta con el valor-p?

El valor-p solo dice **si** hay diferencia. El tamaño del efecto dice **cuán grande** es esa diferencia. En Big Data, casi todo es significativo — necesitas saber si la diferencia *importa*.

### Métricas de tamaño del efecto

**Cohen's d** (para diferencias de medias):
$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$$

| d | Interpretación |
|---|----------------|
| 0.2 | Efecto pequeño |
| 0.5 | Efecto mediano |
| 0.8 | Efecto grande |

**η² (Eta-cuadrado)** para ANOVA: proporción de varianza explicada por el grupo

**Cramér's V** para Chi-cuadrado: asociación entre variables categóricas

In [ ]:
# ── Tamaño del efecto: Cohen's d y la paradoja del n grande ───────────────
def cohens_d(x1, x2):
    """Calcula Cohen's d para dos grupos independientes."""
    n1, n2 = len(x1), len(x2)
    s1, s2 = np.std(x1, ddof=1), np.std(x2, ddof=1)
    s_pooled = np.sqrt(((n1 - 1)*s1**2 + (n2 - 1)*s2**2) / (n1 + n2 - 2))
    return (np.mean(x1) - np.mean(x2)) / s_pooled

def interpretar_d(d):
    d = abs(d)
    if d < 0.2:   return "Negligible"
    elif d < 0.5: return "Pequeño"
    elif d < 0.8: return "Mediano"
    else:         return "Grande"

# Escenario: la misma diferencia real, distintos tamaños de muestra
diferencia_real = 2.5
sigma_comun     = 10.0
tamaños         = [30, 100, 500, 2000, 10000]

print("─" * 70)
print(f"Diferencia real: {diferencia_real} puntos  |  σ = {sigma_comun}  |  d = {diferencia_real/sigma_comun:.2f} (pequeño)")
print("─" * 70)
print(f"{'n':>8} {'p-value':>12} {'Cohen d':>10} {'Efecto':>12} {'Decisión'}")
print("-" * 70)

p_vals, d_vals = [], []
for n in tamaños:
    g1 = np.random.normal(50,                   sigma_comun, n)
    g2 = np.random.normal(50 + diferencia_real, sigma_comun, n)
    _, p = ttest_ind(g1, g2)
    d    = cohens_d(g1, g2)
    p_vals.append(p); d_vals.append(d)
    decision = "✅ Signif." if p < 0.05 else "❌ No signif."
    print(f"{n:>8} {p:>12.6f} {d:>10.3f} {interpretar_d(d):>12} {decision}")

# ── Visualización: p-value y Cohen's d vs n ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogx(tamaños, p_vals, 'o-', color='steelblue', lw=2, ms=8)
ax1.axhline(0.05, color='crimson', linestyle='--', lw=2, label='α = 0.05')
ax1.set_title('p-value decrece con n\n(misma diferencia real)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tamaño de muestra (log)'); ax1.set_ylabel('p-value')
ax1.legend()

ax2.semilogx(tamaños, [abs(d) for d in d_vals], 's-', color='tomato', lw=2, ms=8)
for nivel, label, color in [(0.2, 'pequeño', 'green'), (0.5, 'mediano', 'orange'), (0.8, 'grande', 'red')]:
    ax2.axhline(nivel, color=color, linestyle=':', lw=1.5, alpha=0.8, label=f'd={nivel} ({label})')
ax2.set_title("Cohen's d es estable\n(no depende de n)", fontsize=12, fontweight='bold')
ax2.set_xlabel('Tamaño de muestra (log)'); ax2.set_ylabel("Cohen's d")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Lección clave para Data Scientists:")
print("   Con n=10,000 encontramos diferencias 'significativas' que son NEGLIGIBLES (d=0.25)")
print("   Reporta SIEMPRE el tamaño del efecto junto al p-value")

---
## 13. Bootstrap Hypothesis Testing — Inferencia sin Distribuciones

### 📖 ¿Por qué Bootstrap?

Las pruebas clásicas asumen una distribución (normal, t, chi-cuadrado). **Bootstrap** usa los datos mismos para construir la distribución nula por remuestreo, sin ningún supuesto distribucional.

### Algoritmo

```
1. Calcula el estadístico observado: T_obs = f(datos)
2. Para i = 1 a B (ej. 10,000):
   a. Mezcla/remuestrea los datos bajo H₀
   b. Calcula T_i = f(datos_remuestreados)
3. p-value = P(T_i ≥ T_obs) ≈ proporción de T_i más extremos que T_obs
```

### Ventajas sobre pruebas clásicas

- ✅ No requiere normalidad ni ninguna distribución específica
- ✅ Funciona con **cualquier estadístico** (mediana, percentil 90, ratio...)
- ✅ Ideal para métricas de ML (AUC, F1, RMSE)
- ⚠️ Más costoso computacionalmente (pero trivial hoy)

In [ ]:
# ── Bootstrap Test: ¿Hay diferencia en la mediana del ingreso? ────────────
# Datos sesgados (no normales): ingresos mensuales (en miles)
np.random.seed(5)
grupo_control    = np.random.lognormal(mean=3.0, sigma=0.5, size=60)
grupo_tratamiento = np.random.lognormal(mean=3.2, sigma=0.5, size=60)

# Estadístico observado: diferencia de medianas
diff_mediana_obs = np.median(grupo_tratamiento) - np.median(grupo_control)

# Bootstrap bajo H₀: unir ambos grupos y remuestrear
todos     = np.concatenate([grupo_control, grupo_tratamiento])
n_control = len(grupo_control)
B         = 10_000
diffs_bootstrap = np.zeros(B)

for i in range(B):
    perm          = np.random.permutation(todos)
    boot_ctrl     = perm[:n_control]
    boot_trat     = perm[n_control:]
    diffs_bootstrap[i] = np.median(boot_trat) - np.median(boot_ctrl)

# p-value bilateral
p_bootstrap = np.mean(np.abs(diffs_bootstrap) >= np.abs(diff_mediana_obs))

# Comparación con Mann-Whitney (no paramétrico clásico)
_, p_mw = mannwhitneyu(grupo_tratamiento, grupo_control, alternative='two-sided')

print("=" * 60)
print("Bootstrap Test — Diferencia en mediana de ingresos")
print("=" * 60)
print(f"  Grupo Control    : mediana = {np.median(grupo_control):.2f}k")
print(f"  Grupo Tratamiento: mediana = {np.median(grupo_tratamiento):.2f}k")
print(f"  Diferencia obs.  : {diff_mediana_obs:.2f}k")
print(f"\n  H₀: mediana_A = mediana_B")
print(f"  H₁: mediana_A ≠ mediana_B")
print(f"\n  Bootstrap p-value  : {p_bootstrap:.4f} (B = {B:,})")
print(f"  Mann-Whitney p-val : {p_mw:.4f}")
print(f"\n  {'✅ Rechazamos H₀: diferencia significativa' if p_bootstrap < 0.05 else '❌ No rechazamos H₀'}")

# ── Visualización ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(grupo_control,     bins=25, alpha=0.6, color='steelblue', label='Control', density=True)
ax1.hist(grupo_tratamiento, bins=25, alpha=0.6, color='tomato',    label='Tratamiento', density=True)
ax1.axvline(np.median(grupo_control),     color='steelblue', lw=2.5, linestyle='--')
ax1.axvline(np.median(grupo_tratamiento), color='tomato',    lw=2.5, linestyle='--')
ax1.set_title('Distribución de ingresos\n(datos sesgados → no usar t-test)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Ingreso mensual (miles)'); ax1.legend()

ax2.hist(diffs_bootstrap, bins=60, color='steelblue', alpha=0.7, density=True,
         label=f'Distribución bootstrap\nnula (B={B:,})')
ax2.axvline( diff_mediana_obs, color='crimson', lw=2.5, linestyle='--',
             label=f'Diferencia obs. = {diff_mediana_obs:.2f}')
ax2.axvline(-diff_mediana_obs, color='crimson', lw=2.5, linestyle='--')
ax2.fill_between(
    np.sort(diffs_bootstrap),
    0, [1/len(diffs_bootstrap)*100]*len(diffs_bootstrap),  # placeholder density
)
ax2.set_title(f'Bootstrap — Distribución nula\np = {p_bootstrap:.4f}', fontsize=12, fontweight='bold')
ax2.set_xlabel('Diferencia de medianas (bootstrap)'); ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 🏆 Resumen Final — Guía Rápida de Decisión

### ¿Qué prueba usar en cada situación?

```
DATOS CONTINUOS
│
├── 1 grupo vs valor de referencia
│   ├── σ conocida o n > 30   →  Prueba Z
│   └── σ desconocida         →  Prueba t (una muestra)
│       └── No normal, n<30   →  Wilcoxon signed-rank
│
├── 2 grupos independientes
│   ├── Normales, var. iguales →  Prueba t (Student)
│   ├── Normales, var. desig.  →  Prueba t (Welch)
│   └── No normales            →  Mann-Whitney U
│
├── 2 grupos pareados/relacionados
│   ├── Normal                 →  Prueba t pareada
│   └── No normal              →  Wilcoxon signed-rank
│
└── 3+ grupos
    ├── Normales, var. iguales →  ANOVA de una vía + post-hoc Tukey
    └── No normales            →  Kruskal-Wallis + post-hoc Dunn

DATOS CATEGÓRICOS
├── Independencia de 2 var.    →  Chi-cuadrado (o Fisher si n<5 por celda)
└── Bondad de ajuste           →  Chi-cuadrado de bondad de ajuste

¿NINGUNA DISTRIBUCIÓN ASUMIBLE?   →  Bootstrap permutation test
```

### Los 6 mandamientos del Data Scientist estadístico

| # | Mandamiento |
|---|-------------|
| 1 | Siempre verifica los supuestos antes de aplicar una prueba |
| 2 | Reporta el tamaño del efecto junto al p-value |
| 3 | Con Big Data, p < 0.05 no implica importancia práctica |
| 4 | Corrige por comparaciones múltiples (Bonferroni o FDR) |
| 5 | Define α antes de ver los datos, no después |
| 6 | La ausencia de significancia ≠ prueba de la nulidad |

In [ ]:
# ── Dashboard final: resumen visual de todas las pruebas ──────────────────
pruebas = {
    'Prueba Z':          {'categoria': 'Paramétrica', 'uso': '1 grupo, σ conocida',   'supuesto_normalidad': True},
    'Prueba t (1 samp)': {'categoria': 'Paramétrica', 'uso': '1 grupo, σ desconocida','supuesto_normalidad': True},
    'Prueba t (2 samp)': {'categoria': 'Paramétrica', 'uso': '2 grupos ind.',          'supuesto_normalidad': True},
    'Prueba t (par.)':   {'categoria': 'Paramétrica', 'uso': '2 grupos pareados',     'supuesto_normalidad': True},
    'ANOVA':             {'categoria': 'Paramétrica', 'uso': '3+ grupos',              'supuesto_normalidad': True},
    'Chi-cuadrado':      {'categoria': 'No param.',   'uso': 'Variables cat.',         'supuesto_normalidad': False},
    'Mann-Whitney U':    {'categoria': 'No param.',   'uso': '2 grupos ind.',          'supuesto_normalidad': False},
    'Wilcoxon SR':       {'categoria': 'No param.',   'uso': '1/2 grupos pareados',   'supuesto_normalidad': False},
    'Kruskal-Wallis':    {'categoria': 'No param.',   'uso': '3+ grupos',              'supuesto_normalidad': False},
    'Bootstrap Perm.':   {'categoria': 'Remuestreo',  'uso': 'Cualquier estadístico', 'supuesto_normalidad': False},
}

df_resumen = pd.DataFrame(pruebas).T.reset_index().rename(columns={'index': 'Prueba'})

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
colores_header = ['#2c3e50'] * 4
colores_filas  = ['#ecf0f1' if i % 2 == 0 else '#ffffff' for i in range(len(df_resumen))]

tabla_vis = ax.table(
    cellText=df_resumen.values,
    colLabels=['Prueba', 'Categoría', 'Uso principal', 'Supone normalidad'],
    cellLoc='left', loc='center',
    bbox=[0, 0, 1, 1]
)
tabla_vis.auto_set_font_size(False)
tabla_vis.set_fontsize(11)

for (row, col), cell in tabla_vis.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row > 0:
        categoria = df_resumen.iloc[row-1]['categoria']
        if categoria == 'Paramétrica':
            cell.set_facecolor('#d5e8f7')
        elif categoria == 'No param.':
            cell.set_facecolor('#d5f7e8')
        else:
            cell.set_facecolor('#f7f0d5')
    cell.set_edgecolor('#bdc3c7')

plt.title('Guía Rápida: Selección de Prueba de Hipótesis',
          fontsize=14, fontweight='bold', pad=15, y=1.02)
plt.tight_layout()
plt.show()

print("\n✅ Notebook completado — Pruebas de Hipótesis y sus Aplicaciones en ML")
print("─" * 60)
print("Conceptos cubiertos:")
print("  1. Lógica de H₀ / H₁ y zonas de rechazo")
print("  2. Errores Tipo I (α), Tipo II (β) y Poder estadístico")
print("  3. Valor-p: qué significa y qué NO significa")
print("  4. Prueba Z y prueba t (3 variantes)")
print("  5. Chi-cuadrado para variables categóricas")
print("  6. ANOVA + post-hoc con corrección de Bonferroni")
print("  7. Pruebas no paramétricas (Mann-Whitney, Wilcoxon)")
print("  8. Feature selection y comparación de modelos con hipótesis")
print("  9. A/B Testing en producción")
print(" 10. Corrección por comparaciones múltiples (Bonferroni + FDR)")
print(" 11. Tamaño del efecto (Cohen's d, η², Cramér's V)")
print(" 12. Bootstrap permutation test")

---
## 🎯 Ejercicios Prácticos

Resuelve estos ejercicios para afianzar los conceptos:

### Ejercicio 1 — Selección de prueba
Para cada situación, indica qué prueba usarías y por qué:
- a) Quieres saber si el tiempo de respuesta de tu API (no normal, muy sesgado) es menor en la versión nueva que en la anterior (muestras independientes).
- b) Tienes 5 variantes de un modelo y quieres saber si alguna difiere en AUC.
- c) Analizas si el tipo de dispositivo (móvil, desktop, tablet) está relacionado con la tasa de abandono del carrito.
- d) Mides el accuracy de tu modelo antes y después de agregar una feature nueva, en los mismos 20 folds.

### Ejercicio 2 — Interpretación
Un equipo reporta: *"Probamos 200 features y encontramos 12 significativas (p < 0.05)".*
- a) ¿Cuántas podrían ser falsos positivos sin corrección?
- b) Aplica corrección FDR y compara con Bonferroni.

### Ejercicio 3 — A/B Test completo
Datos: Control: 3,200 conversiones de 40,000 usuarios. Tratamiento: 3,520 de 40,000.
- a) Formula H₀ y H₁
- b) Calcula el estadístico Z y el p-value manualmente
- c) Calcula el IC 95% de la diferencia
- d) ¿Es el efecto estadísticamente significativo Y prácticamente relevante?

### Ejercicio 4 — Bootstrap
Tienes dos grupos de scores de churn (distribución bimodal, no normal).  
Implementa un bootstrap permutation test para probar si la diferencia de medias es significativa.

---
> 📚 **Recursos recomendados:**
> - *Statistics for Data Scientists* — Peter Bruce & Andrew Bruce
> - *Naked Statistics* — Charles Wheelan (lectura ligera, muy recomendada)
> - Khan Academy: Statistics and Probability
> - SciPy documentation: `scipy.stats`